# ShapeGrammar in TopologicPy

This notebook introduces the public `ShapeGrammar` API for rule-based geometric transformation in TopologicPy.

A shape grammar contains **rules**. Each rule has:

- an **input topology** that acts as the geometric pattern,
- an optional **output topology** or Boolean tool,
- an **operation** such as Replace, Difference, Union, Transform, or Divide,
- an optional rule-local transformation matrix,
- optional metadata.

The workflow is:

1. author rules in a local coordinate frame;
2. compile/index the grammar;
3. find rules applicable to a target topology;
4. apply a selected match;
5. inspect derivation history and exact geometric lineage.

With the PythonOCC/OCCT 8 backend, Boolean and transform lineage uses the same exact native history path used elsewhere in TopologicPy, with BRepGraph used internally for final-result membership. No OCCT or BRepGraph objects are exposed through the `ShapeGrammar` API.

## 1. Imports

This tutorial assumes a recent TopologicPy version containing the refactored public `ShapeGrammar` class.

For exact BRepGraph-backed provenance, use the PythonOCC/OCCT 8 backend. The grammar itself remains engine-neutral.

In [ ]:
from collections import Counter

from topologicpy.ShapeGrammar import ShapeGrammar
from topologicpy.Topology import Topology
from topologicpy.Vertex import Vertex
from topologicpy.Cell import Cell
from topologicpy.Matrix import Matrix
from topologicpy.Plotly import Plotly
from topologicpy.TGraph import TGraph

## 2. Small visualization helpers

The helpers below use the existing TopologicPy Plotly pathway. They do not modify grammar state.

In [ ]:
def show_topology(topology, width=900, height=600):
    data = Plotly.DataByTopology(topology)
    fig = Plotly.FigureByData(data, width=width, height=height)
    fig.show()
    return fig


def show_rule(grammar, rule_id):
    fig = grammar.FigureByRule(rule_id, silent=True)
    if fig is not None:
        fig.show()
    return fig

## 3. Create a grammar

`ShapeGrammar` assigns every rule a stable integer identifier. Rule IDs are the preferred way to store and refer to rules in downstream code.

In [ ]:
grammar = ShapeGrammar(
    title="Room transformation grammar",
    description="A compact tutorial grammar demonstrating matching, application, caching, derivations, and lineage.",
)

grammar.OperationTitles()

The public operation vocabulary includes replacement, Boolean operations, local transforms, and regular subdivision. Operation lookup is case-insensitive.

In [ ]:
grammar.OperationByTitle("difference")

## 4. Author rule geometry

Rules are authored in a **rule-local coordinate frame**.

Here the pattern is a simple rectangular room module. We also attach a semantic dictionary entry (`program="room"`), which can later participate in matching.

The target is a translated and rotated copy of the pattern with its own instance dictionary. Geometry determines similarity; selected dictionary keys can additionally constrain rule applicability.

In [ ]:
pattern_room = Cell.Prism(
    origin=Vertex.Origin(),
    width=4.0,
    length=3.0,
    height=3.0,
    placement="center",
    silent=True,
)
pattern_room = Topology.SetDictionary(
    pattern_room,
    {"program": "room", "family": "module_A"},
    silent=True,
)

# A competing semantic pattern with identical geometry.
pattern_service = Cell.Prism(
    origin=Vertex.Origin(),
    width=4.0,
    length=3.0,
    height=3.0,
    placement="center",
    silent=True,
)
pattern_service = Topology.SetDictionary(
    pattern_service,
    {"program": "service", "family": "module_A"},
    silent=True,
)

# Target instance: geometrically similar, but translated and rotated.
target = Topology.Rotate(
    pattern_room,
    origin=Topology.Centroid(pattern_room),
    axis=[0, 0, 1],
    angle=20,
    silent=True,
)
target = Topology.Translate(target, 7.0, 2.0, 0.5, silent=True)
target = Topology.SetDictionary(
    target,
    {"program": "room", "instance": "R-101"},
    silent=True,
)

show_topology(target)

### Rule output/tool geometry

For a Difference rule, the rule output is interpreted as the Boolean tool authored in the same local frame as the rule input.

The cylinder below passes through the room along its local X axis. During application, ShapeGrammar maps it into the target's matched coordinate frame before executing the Boolean operation.

In [ ]:
opening_tool = Cell.Cylinder(
    origin=Vertex.Origin(),
    radius=0.65,
    height=5.0,
    uSides=32,
    direction=[1, 0, 0],
    placement="center",
    silent=True,
)

replacement = Cell.Prism(
    origin=Vertex.Origin(),
    width=3.0,
    length=3.0,
    height=4.0,
    placement="center",
    silent=True,
)

service_replacement = Cell.Cylinder(
    origin=Vertex.Origin(),
    radius=1.4,
    height=3.0,
    uSides=24,
    placement="center",
    silent=True,
)

## 5. Add rules

This grammar contains several rules sharing the same geometric pattern. Because rules are indexed by topology type and structural counts, obviously incompatible rules are rejected before the more expensive similarity calculation.

`metadata` is descriptive rule metadata. It is separate from topology dictionaries used by semantic matching.

In [ ]:
cut_rule = grammar.AddRule(
    pattern_room,
    opening_tool,
    title="Cut axial opening",
    description="Subtract a cylindrical opening from a room module.",
    operation="Difference",
    metadata={"category": "subtractive"},
)

replace_rule = grammar.AddRule(
    pattern_room,
    replacement,
    title="Replace with tall module",
    operation="Replace",
    metadata={"category": "replacement"},
)

divide_rule = grammar.AddRule(
    pattern_room,
    title="Divide room 2 x 2",
    operation="Divide",
    uSides=2,
    vSides=2,
    wSides=1,
    metadata={"category": "subdivision"},
)

lift_rule = grammar.AddRule(
    pattern_room,
    title="Lift locally",
    operation="Transform",
    matrix=Matrix.ByTranslation(translateX=0, translateY=0, translateZ=1.5),
    metadata={"category": "transform"},
)

service_rule = grammar.AddRule(
    pattern_service,
    service_replacement,
    title="Service replacement",
    operation="Replace",
    metadata={"category": "replacement"},
)

grammar.Rules()

## 6. Compile the grammar

Compilation builds the internal rule index and precomputes data used by matching and repeated application. It is safe to call explicitly, although matching also compiles lazily when required.

In [ ]:
grammar.Compile()

## 7. Find applicable rules

### Pure geometric matching

Without semantic keys, the room and service patterns are geometrically identical, so both can match.

In [ ]:
geometry_matches = grammar.ApplicableRules(target, silent=True)

[(m["rule"], m["title"], m["operation"]) for m in geometry_matches]

### Geometry + semantic matching

Passing `keys=["program"]` requires the target and rule input dictionaries to agree on that key. The service rule is therefore removed before the expensive geometric similarity test.

The cell deliberately repeats the identical semantic query once so the runtime diagnostics also demonstrate a genuine **match-cache hit**.

In [ ]:
matches = grammar.ApplicableRules(
    target,
    keys=["program"],
    silent=True,
)

# Repeat the identical query to demonstrate the match cache.
cached_matches = grammar.ApplicableRules(
    target,
    keys=["program"],
    silent=True,
)

{
    "applicable": [(m["rule"], m["title"], m["operation"]) for m in matches],
    "match_cache_hits": grammar.Status()["matchCacheHits"],
    "match_cache_misses": grammar.Status()["matchCacheMisses"],
}

Each match descriptor contains:

- the stable rule ID,
- title and operation,
- the 4×4 similarity transform from rule input to target,
- rule metadata.

That descriptor can be passed directly to `ApplyMatch`.

In [ ]:
cut_match = next(m for m in matches if m["rule"] == cut_rule)
cut_match

## 8. Preview a rule

`FigureByRule` evaluates a preview without adding a user application to the derivation history. The figure places the rule input and its local result side by side.

In [ ]:
show_rule(grammar, cut_rule)

## 9. Apply a matched rule

For Boolean rules, ShapeGrammar operates on the **actual target topology**. The rule output/tool is first transformed into the matched target frame, then the Boolean is executed.

This is important: the target is not merely a transformed copy of a Boolean performed on the stored pattern.

In [ ]:
cut_result = grammar.ApplyMatch(
    target,
    cut_match,
    silent=True,
)

application = grammar.Application()
application

Visualize the target and resulting topology using the grammar's input/output comparison helper.

In [ ]:
fig = grammar.FigureByInputOutput(target, cut_result, silent=True)
if fig is not None:
    fig.show()

## 10. Application caching

Applying the same rule to the same target with the same match matrix reuses the application cache. A new application record is still created, so the derivation history remains explicit.

In [ ]:
cached_result = grammar.ApplyMatch(
    target,
    cut_match,
    silent=True,
)

grammar.Application()

In [ ]:
status = grammar.Status()
status

Useful diagnostic fields include:

- `matchCacheHits` / `matchCacheMisses`
- `applyCacheHits` / `applyCacheMisses`
- `similarityTests`
- `applications`
- `lineageRecords`
- `usedBRepGraph`

The semantic matching section deliberately repeated one identical query, so `matchCacheHits` should now be at least 1. On PythonOCC/OCCT 8, `usedBRepGraph=True` confirms that at least one captured history stage used the private BRepGraph result-membership index. The operation-specific lineage inspection below verifies that the **Difference stage itself** did so.

## 11. Exact lineage

Each rule application can record source-to-result relationships such as:

- `modified`
- `generated`
- `unchanged`
- `deleted`

A Boolean ShapeGrammar application can contain more than one captured stage. In this example the cylindrical rule tool is first transformed into the matched target frame (`GTransform`), then the actual Boolean cut runs (`Difference`). Inspecting lineage by **operation and relation** makes those stages explicit.

The records are backend-neutral dictionaries containing TopologicPy topology objects rather than OCCT or BRepGraph identifiers.

In [ ]:
# Inspect the first (cold) application rather than the cached replay.
app_id = application["application"]
history = grammar.History(application=app_id)

difference_history = [
    record
    for record in history
    if record.get("operation") == "Difference"
]

{
    "by_operation_and_relation": Counter(
        (record.get("operation"), record.get("relation"))
        for record in history
    ),
    "difference_records": len(difference_history),
    "difference_uses_brepgraph": any(
        record.get("usedBRepGraph") is True
        for record in difference_history
    ),
}

In [ ]:
# Inspect a compact sample from the Boolean itself.
[
    {
        "relation": r.get("relation"),
        "sourceRole": r.get("sourceRole"),
        "sourceType": r.get("sourceType"),
        "resultType": r.get("resultType"),
        "operation": r.get("operation"),
        "usedBRepGraph": r.get("usedBRepGraph"),
    }
    for r in difference_history[:10]
]

Convenience queries return the actual derived topologies.

`GeneratedBy`, `ModifiedBy`, and `DeletedBy` are relation-oriented queries across the whole application. For a Boolean, **do not assume that a new interface Face must be classified as `generated`**: OCCT may legitimately report it as `modified` depending on the operation and native history. For documentation and diagnostics, the operation-aware `difference_history` above is therefore the authoritative view of the Boolean stage.

In [ ]:
generated_faces = grammar.GeneratedBy(app_id, topologyType="Face")
modified_faces = grammar.ModifiedBy(app_id, topologyType="Face")
deleted_faces = grammar.DeletedBy(app_id, topologyType="Face")

difference_face_records = [
    record
    for record in difference_history
    if record.get("resultType") == "Face"
    and record.get("result") is not None
]

lineage_face = (
    difference_face_records[0]["result"]
    if difference_face_records
    else None
)

{
    "application_face_queries": {
        "generated": len(generated_faces),
        "modified": len(modified_faces),
        "deleted": len(deleted_faces),
    },
    "difference_face_relations": Counter(
        record.get("relation")
        for record in difference_face_records
    ),
    "difference_result_faces": len(difference_face_records),
}

### Trace a result backwards

`Origins` traces a derived topology back through all captured stages in the application. We select a Face that actually appears in the **Difference** result lineage, regardless of whether OCCT classified it as `modified` or `generated`. This allows the trace to pass through the aligned tool geometry and, where appropriate, back to the original rule output.

In [ ]:
origins = (
    grammar.Origins(lineage_face, application=app_id)
    if lineage_face is not None
    else []
)

origins

## 12. Lineage graph

`LineageGraph` converts exact subtopology provenance into a derived `TGraph`. This graph is distinct from the grammar's rule-level derivation graph.

Because one ShapeGrammar application may contain several native stages, the graph can contain edges for both the tool alignment (`GTransform`) and the Boolean (`Difference`). The second cell below filters the edge dictionaries to the Boolean stage explicitly.

In [ ]:
lineage_graph = grammar.LineageGraph(application=app_id)

lineage_vertices = TGraph.Vertices(lineage_graph)
lineage_edges = TGraph.Edges(lineage_graph)

{
    "vertices": len(lineage_vertices),
    "edges": len(lineage_edges),
}

In [ ]:
# Edge dictionaries carry relation, operation, rule, and application metadata.
difference_edges = [
    edge.get("dictionary", {})
    for edge in lineage_edges
    if edge.get("dictionary", {}).get("operation") == "Difference"
]

difference_edges[:10]

## 13. Apply other rule types

The same target can be processed by other rules using their match descriptors.

### Divide

In [ ]:
divide_match = next(m for m in matches if m["rule"] == divide_rule)
divided = grammar.ApplyMatch(target, divide_match, silent=True)

fig = grammar.FigureByInputOutput(target, divided, silent=True)
if fig is not None:
    fig.show()

### Transform

A Transform rule is authored in the rule-local frame. ShapeGrammar conjugates the local transformation through the match matrix, so the transform follows the orientation and scale of the matched target.

In [ ]:
lift_match = next(m for m in matches if m["rule"] == lift_rule)
lifted = grammar.ApplyMatch(target, lift_match, silent=True)

fig = grammar.FigureByInputOutput(target, lifted, silent=True)
if fig is not None:
    fig.show()

## 14. Derivation graph

`DerivationGraph` records topology states and the rule applications connecting them. It answers a different question from `LineageGraph`:

- **DerivationGraph**: which rule transformed one design state into another?
- **LineageGraph**: which source subtopologies generated, modified, survived, or disappeared in a particular operation?

In [ ]:
derivation = grammar.DerivationGraph()

derivation_vertices = TGraph.Vertices(derivation)
derivation_edges = TGraph.Edges(derivation)

{
    "states": len(derivation_vertices),
    "applications": len(derivation_edges),
}

In [ ]:
[
    edge.get("dictionary", {})
    for edge in derivation_edges
]

## 15. Persistence

Runtime caches, application history, and native lineage are intentionally **not** persisted. The grammar definition is.

`JSONString()` serializes rule geometry to BREP only at the persistence boundary. A loaded grammar recompiles its runtime state lazily.

In [ ]:
grammar_json = grammar.JSONString(indent=2)

print(grammar_json[:800] + "\n...")

In [ ]:
restored = ShapeGrammar.ByJSONString(grammar_json, silent=True)

{
    "title": restored.title,
    "description": restored.description,
    "rules": len(restored.Rules()),
    "compile": restored.Compile(),
}

## 16. Runtime control

`ClearRuntime()` clears caches and diagnostics while retaining application history by default.

Use `clearHistory=True` when you want a completely fresh derivation run.

`Invalidate()` should be called after externally mutating rule topology dictionaries or other rule state that affects matching.

In [ ]:
grammar.ClearRuntime(clearHistory=False)
grammar.Status()

## 17. Minimal pattern

The essential public workflow is deliberately small:

In [ ]:
g = ShapeGrammar()

pattern = Cell.Prism(width=4, length=3, height=3, silent=True)
tool = Cell.Cylinder(radius=0.6, height=5, direction=[1, 0, 0], silent=True)

rule_id = g.AddRule(
    pattern,
    tool,
    operation="Difference",
    title="Opening",
)

target = Topology.Translate(pattern, 8, 0, 0, silent=True)

match = g.Match(target, silent=True)[0]
result = g.ApplyMatch(target, match, silent=True)

print(g.Status())

## Summary

The public `ShapeGrammar` model separates five concerns cleanly:

1. **Rule definition** — stable rule IDs, local input/output geometry, operation and metadata.
2. **Matching** — structural indexing and optional semantic keys before geometric similarity testing.
3. **Application** — target-based rule execution with match/application caching.
4. **Derivation** — explicit topology-state history through `DerivationGraph`.
5. **Provenance** — exact subtopology lineage through `History`, lineage queries, and `LineageGraph`.

The class remains backend-neutral at the public API while taking advantage of PythonOCC/OCCT history and BRepGraph internally when available.